In [2]:
import sys
import os
from pathlib import Path
from dotenv import load_dotenv
import pandas as pd
import optuna
import importlib

sys.path.append(os.path.abspath(".."))

import src.models.tabnet.tabnet_cv_trainer as cv
import src.models.tabnet.tabnet_objective as ob
import src.utils.run_optuna as op

In [3]:
# Load data
env_path = Path.cwd().parent / ".env"
load_dotenv(dotenv_path=env_path)
url = os.environ.get("OPTUNA_STORAGE_URL")

tr_df8 = pd.read_parquet("../artifacts/features/base/tr_df8.parquet")
test_df8 = pd.read_parquet("../artifacts/features/base/test_df8.parquet")

l1_tr_df2 = pd.read_parquet("../artifacts/features/l1/l1_tr_df2.parquet")

In [14]:
importlib.reload(cv)
trainer = cv.TabNetCVTrainer()
trainer.fit(tr_df8, test_df8)


Fold 1


/home/hanse/miniconda3/envs/torch22/lib/python3.11/site-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cuda
  warnings.warn(f"Device used : {self.device}")
/home/hanse/miniconda3/envs/torch22/lib/python3.11/site-packages/torch/optim/lr_scheduler.py:28: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn("The verbose parameter is deprecated. Please use get_last_lr() "


epoch 0  | loss: 0.21343 | val_0_logloss: 0.17715 |  0:00:28s
epoch 10 | loss: 0.16828 | val_0_logloss: 0.16182 |  0:05:13s
epoch 20 | loss: 0.15801 | val_0_logloss: 0.15612 |  0:10:00s
epoch 30 | loss: 0.15339 | val_0_logloss: 0.15374 |  0:14:45s
epoch 40 | loss: 0.14958 | val_0_logloss: 0.15497 |  0:19:44s
epoch 50 | loss: 0.14736 | val_0_logloss: 0.15415 |  0:24:34s

Early stopping occurred at epoch 54 with best_epoch = 34 and best_val_0_logloss = 0.15363


/home/hanse/miniconda3/envs/torch22/lib/python3.11/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


Best Logloss: 0.96332
Training time: 00:27:22

Fold 2


/home/hanse/miniconda3/envs/torch22/lib/python3.11/site-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cuda
  warnings.warn(f"Device used : {self.device}")
/home/hanse/miniconda3/envs/torch22/lib/python3.11/site-packages/torch/optim/lr_scheduler.py:28: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn("The verbose parameter is deprecated. Please use get_last_lr() "


epoch 0  | loss: 0.21029 | val_0_logloss: 0.17266 |  0:00:29s


KeyboardInterrupt: 

In [ ]:
# Tuning
importlib.reload(cv)
importlib.reload(ob)
objective = ob.create_objective(
    tr_df8,
    early_stopping_rounds=10,
    min_epochs=10,
)

op.run_optuna_search(
    objective,
    n_trials=20,
    direction="minimize",
    study_name="l1_mlp_v2",
    storage=url,
    sampler=optuna.samplers.TPESampler(n_startup_trials=10, seed=42),
)